# Разведочный анализ данных (EDA) — прогноз дохода клиентов банка

Этот блокнот показывает исходные данные и те находки, которые определили наше решение:
почему метрика заставляет смотреть на богатых клиентов, где живёт ошибка, и почему мы
выбрали именно такую модель и приёмы. Каждый график сохраняется в папку `charts/` —
их можно вставлять в презентацию.

Порядок находок:
1. Распределение дохода — тяжёлый правый хвост.
2. Веса метрики — U-образные (главная находка).
3. Декомпозиция ошибки — 71% в верхнем сегменте.
4. Прямая зарплата почти равна доходу, но есть лишь у части клиентов.
5. Сильные признаки дохода сильно разрежены — поэтому нужна модель.

# Настройка и загрузка

In [ ]:
import pandas as pd, numpy as np, os
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':12,'axes.edgecolor':'#cbd5e1',
  'axes.linewidth':1,'figure.dpi':130})
BLUE='#2563EB'; GREY='#c9d6ea'; INK='#0f1729'; MUT='#6b7280'; GREEN='#15803d'
os.makedirs('charts', exist_ok=True)
def style(ax):
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.tick_params(colors=MUT)
def title(ax,t,sub=None):
    ax.set_title(t,loc='left',fontsize=14,fontweight='bold',color=INK,pad=14)
    if sub: ax.text(0,1.02,sub,transform=ax.transAxes,fontsize=10,color=MUT,va='bottom')

train = pd.read_csv('train.csv', decimal=',', sep=';', low_memory=False)
test  = pd.read_csv('test.csv',  decimal=',', sep=';', low_memory=False)
print('train:', train.shape, '| test:', test.shape)
y = train['target'].values.astype(float); w = train['w'].values.astype(float)

## 1. Распределение дохода: тяжёлый правый хвост
Доход скошен вправо — медиана заметно ниже среднего, есть длинный хвост высокодоходных.
Это типичная ловушка для деревьев: они «сжимают» хвост к среднему и занижают богатых.

In [ ]:
print('медиана=%.0f  среднее=%.0f  максимум=%.0f' % (np.median(y), y.mean(), y.max()))
fig,ax=plt.subplots(figsize=(7,4.2))
ax.hist(np.clip(y,0,400000),bins=60,color=BLUE,alpha=0.85,edgecolor='white',linewidth=0.3)
ax.axvline(np.median(y),color=INK,ls='--',lw=1.5); ax.axvline(y.mean(),color=GREEN,ls='--',lw=1.5)
ax.text(np.median(y)+4000,ax.get_ylim()[1]*0.9,'медиана 63k',color=INK,fontsize=10)
ax.text(y.mean()+4000,ax.get_ylim()[1]*0.78,'среднее 98k',color=GREEN,fontsize=10)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x,_:f'{int(x/1000)}k'))
ax.set_xlabel('доход, ₽'); ax.set_ylabel('клиентов'); style(ax)
title(ax,'Доход сильно скошен вправо','тяжёлый хвост высокодоходных (обрезано на 400k)')
plt.tight_layout(); plt.savefig('charts/01_target_distribution.png',bbox_inches='tight'); plt.show()

## 2. Веса метрики — U-образные (главная находка)
У каждого клиента свой вес в метрике WMAE. Разложив средний вес по децилям дохода, видим:
середина почти бесплатна (вес ≈ 0.05), а верхний дециль весит примерно в 40 раз больше.
**Вывод:** оптимизировать среднюю ошибку бессмысленно — надо быть точным на богатых.

In [ ]:
dec = pd.qcut(y,10,labels=False,duplicates='drop')
wm = [w[dec==i].mean() for i in range(dec.max()+1)]
top_share = w[dec==dec.max()].sum()/w.sum()
print('вес верхнего дециля: %.2f | доля всей массы весов: %.1f%%' % (wm[-1], top_share*100))
fig,ax=plt.subplots(figsize=(7,4.2))
cols=[BLUE if (i==0 or i==len(wm)-1) else GREY for i in range(len(wm))]
ax.bar(range(1,len(wm)+1),wm,color=cols,edgecolor='white')
ax.text(len(wm),wm[-1],f' {wm[-1]:.1f}',va='bottom',ha='center',color=BLUE,fontweight='bold')
ax.set_xlabel('дециль дохода (1 — беднейшие, 10 — богатейшие)'); ax.set_ylabel('средний вес')
style(ax); title(ax,'Веса метрики — U-образные','середина почти бесплатна, верхний дециль весит в ~40 раз больше')
plt.tight_layout(); plt.savefig('charts/02_weight_by_decile.png',bbox_inches='tight'); plt.show()

## 3. Декомпозиция ошибки: 71% — в верхнем сегменте
Обучим быструю базовую модель по времени (янв–май → июнь) и разложим взвешенную ошибку
по сегментам дохода. Почти вся ошибка метрики приходится на клиентов 171k+, которых модель
занижает. **Вывод:** улучшать надо точечно верхний сегмент, а не модель «вообще».

In [ ]:
import lightgbm as lgb
CATS=['gender','addrref']; HC=['dp_ewb_last_employment_position','city_smart_name','adminarea']
DROP=['id','target','w','dt','period_last_act_ad']
tr=train.copy()
for c in tr.columns:
    if c not in set(CATS+HC+['id','dt']): tr[c]=pd.to_numeric(tr[c],errors='coerce').astype('float32')
for c in CATS: tr[c]=tr[c].astype('category')
FEATS=[c for c in tr.columns if c not in DROP and c not in HC]
vm=(tr['dt']=='2024-06-30').values; tm=~vm
d=lgb.Dataset(tr.loc[tm,FEATS],tr.loc[tm,'target'],weight=tr.loc[tm,'w'],categorical_feature=CATS)
m=lgb.train(dict(objective='regression_l1',num_leaves=63,learning_rate=0.05,verbose=-1,seed=42),
            d,num_boost_round=600)
yv=tr.loc[vm,'target'].values; wv=tr.loc[vm,'w'].values
pv=np.clip(m.predict(tr.loc[vm,FEATS]),20000,1500000)
seg=pd.cut(yv,[0,40000,120000,171000,1e12],labels=['0–40k','40–120k','120–171k','171k+'])
werr=wv*np.abs(yv-pv); labels=['0–40k','40–120k','120–171k','171k+']
share=[werr[seg==s].sum()/werr.sum()*100 for s in labels]
bias=[np.median((pv-yv)[seg==s]) for s in labels]
print('доля ошибки по сегментам:', dict(zip(labels,[round(s) for s in share])))
print('медианное смещение 171k+: %.0f (недооценка)' % bias[-1])
fig,ax=plt.subplots(figsize=(7,4.2))
b=ax.bar(labels,share,color=[GREY,GREY,GREY,BLUE],edgecolor='white')
for r,s in zip(b,share): ax.text(r.get_x()+r.get_width()/2,s+1,f'{s:.0f}%',ha='center',color=INK,fontweight='bold')
ax.set_ylabel('доля взвешенной ошибки, %'); ax.set_ylim(0,80); style(ax)
title(ax,'71% ошибки — в верхнем сегменте','клиенты 171k+ дают почти всю ошибку метрики')
plt.tight_layout(); plt.savefig('charts/03_error_decomposition.png',bbox_inches='tight'); plt.show()

## 4. Прямая зарплата почти равна доходу — но есть не у всех
Признак `salary_6to12m_avg` там, где он заполнен, почти совпадает с доходом (корреляция 0.935).
Но заполнен он лишь у ~19% клиентов. **Вывод:** для этих клиентов выгодно довериться зарплате
напрямую (приём override), а не разбавленному деревом прогнозу.

In [ ]:
sal=pd.to_numeric(train['salary_6to12m_avg'],errors='coerce').values
m_=~np.isnan(sal)
from scipy.stats import spearmanr
print('заполнено: %.1f%% | корреляция с доходом (Spearman): %.3f' %
      (m_.mean()*100, spearmanr(sal[m_], y[m_]).correlation))
idx=np.random.RandomState(0).choice(np.where(m_)[0],size=min(4000,m_.sum()),replace=False)
fig,ax=plt.subplots(figsize=(7,4.2))
ax.scatter(np.clip(sal[idx],0,400000),np.clip(y[idx],0,400000),s=6,alpha=0.25,color=BLUE,edgecolors='none')
ax.plot([0,400000],[0,400000],color=INK,ls='--',lw=1)
ax.set_xlim(0,400000); ax.set_ylim(0,400000)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x,_:f'{int(x/1000)}k'))
ax.yaxis.set_major_formatter(FuncFormatter(lambda x,_:f'{int(x/1000)}k'))
ax.set_xlabel('salary_6to12m_avg'); ax.set_ylabel('истинный доход'); style(ax)
title(ax,'Где зарплата есть — она почти равна доходу','корреляция 0.935, но заполнена лишь у 19% клиентов')
plt.tight_layout(); plt.savefig('charts/04_salary_vs_income.png',bbox_inches='tight'); plt.show()

## 5. Сильные признаки дохода сильно разрежены
Прямые прокси дохода заполнены лишь у части клиентов, поэтому у большинства доход приходится
восстанавливать по косвенным признакам. **Вывод:** задача нетривиальна и требует модели,
а разреженные сильные признаки нужно уметь использовать точечно.

In [ ]:
keys={'salary_6to12m_avg':'зарплата 6–12м','incomeValue':'incomeValue',
 'dp_ils_avg_salary_1y':'зарплата ИЛС 1г','profit_income_out_rur_amt_12m':'доход-оборот 12м',
 'dp_ewb_last_employment_position':'должность'}
fill=[]
for c in keys:
    if c=='dp_ewb_last_employment_position':
        f=100*(1-train[c].astype(str).isin(['nan','None','']).mean())
    else:
        f=100*(1-pd.to_numeric(train[c],errors='coerce').isna().mean())
    fill.append(f)
order=np.argsort(fill); labs=[list(keys.values())[i] for i in order]; vals=[fill[i] for i in order]
fig,ax=plt.subplots(figsize=(7,4.2))
ax.barh(labs,vals,color=BLUE,edgecolor='white')
for i,v in enumerate(vals): ax.text(v+1,i,f'{v:.0f}%',va='center',color=INK,fontsize=10)
ax.set_xlim(0,100); ax.set_xlabel('% заполнено'); style(ax)
title(ax,'Сильные признаки дохода сильно разрежены','прямая зарплата есть лишь у ~19% — поэтому нужна модель')
plt.tight_layout(); plt.savefig('charts/05_missingness.png',bbox_inches='tight'); plt.show()

## Итог разведочного анализа

Четыре находки задали всё решение:
1. **Тяжёлый хвост** дохода → деревья занижают богатых.
2. **U-образные веса** → метрика штрафует именно за богатых.
3. **71% ошибки в сегменте 171k+** → улучшать надо точечно верх, отсюда сегментная коррекция хвоста.
4. **Разреженная, но почти точная зарплата** → приём override для тех, у кого она есть.

Отсюда выбор модели (LightGBM с L1, согласованной с MAE), ансамбль двух моделей,
сегментная коррекция хвоста и override по прямой зарплате.